# Data Cleaning — Match Goals

In [6]:
import pandas as pd

# Note: keep ALL matches (home + away) — away results needed for form engineering
df = pd.read_csv('../../data/raw/gold_match_goals.csv', sep=',', on_bad_lines='skip')
print(df.shape)
df.head()

(372, 18)


,match_id,match_date,kickoff_time_local,season,stage,competition_name,opponent,period_id,time_min,time_min_sec,is_ohl_goal,scorer_name,assist_player_name,goal_type,home_score_after,away_score_after,var_reviewed,opta_event_id
0,d0te6swsv2y99ywgugc2utbmc,2022-07-23,18:15:00,2022/2023,Regular Season,Belgian Jupiler Pro League,Kortrijk,2,72,71:57,True,M. Maertens,L. Patris,Regular,0,1,NaN,2438482395
1,d0te6swsv2y99ywgugc2utbmc,2022-07-23,18:15:00,2022/2023,Regular Season,Belgian Jupiler Pro League,Kortrijk,2,93,92:03,True,Mousa Tamari,L. Patris,Regular,0,2,NaN,2438491703
2,d256yo3eng04m0fu7b4sl7wno,2022-07-30,18:15:00,2022/2023,Regular Season,Belgian Jupiler Pro League,Westerlo,1,13,12:35,True,J. Þorsteinsson,NaN,Regular,1,0,NaN,2440388221
3,d256yo3eng04m0fu7b4sl7wno,2022-07-30,18:15:00,2022/2023,Regular Season,Belgian Jupiler Pro League,Westerlo,2,77,76:10,True,R. Holzhauser,NaN,Penalty,2,0,NaN,2440445759
4,d3pqkck2grzx98jg4sofhp8us,2022-08-07,21:00:00,2022/2023,Regular Season,Belgian Jupiler Pro League,Antwerp,1,27,26:04,True,N. Nsingi,L. Patris,Regular,0,1,NaN,2443561741


In [7]:
# Fix data types
df['match_date'] = pd.to_datetime(df['match_date'])
df['kickoff_time_local'] = pd.to_datetime(df['kickoff_time_local'], format='%H:%M:%S').dt.time
df['is_ohl_goal'] = df['is_ohl_goal'].map({'true': True, 'false': False}).astype(bool)

In [8]:
# Drop columns with no predictive value
cols_to_drop = [
    'competition_name',     # constant — all Belgian Jupiler Pro League
    'time_min_sec',         # redundant with time_min
    'opta_event_id',        # internal technical ID
    'var_reviewed',         # almost entirely empty
    'scorer_name',          # too granular for attendance prediction
    'assist_player_name',   # too granular + many nulls
]
df = df.drop(columns=cols_to_drop)

print(df.shape)
print(df.dtypes)

(372, 12)
match_id                         str
match_date            datetime64[us]
kickoff_time_local            object
season                           str
stage                            str
opponent                         str
period_id                      int64
time_min                       int64
is_ohl_goal                     bool
goal_type                        str
home_score_after               int64
away_score_after               int64
dtype: object


## Aggregate to match level
One row per match — needed for home form and last result vs opponent engineering.

In [9]:
# Get final score per match (last row per match has the final home/away score)
final_scores = (
    df.sort_values('time_min')
    .groupby('match_id')[['home_score_after', 'away_score_after']]
    .last()
    .rename(columns={'home_score_after': 'goals_home_ft', 'away_score_after': 'goals_away_ft'})
    .reset_index()
)

# Match-level metadata (constant per match — take first row)
match_meta = (
    df.groupby('match_id')
    .first()[['match_date', 'kickoff_time_local', 'season', 'stage', 'opponent']]
    .reset_index()
)

# Goal counts per match
ohl_goals = df[df['is_ohl_goal']].groupby('match_id').size().reset_index(name='ohl_goals_scored')
opp_goals = df[~df['is_ohl_goal']].groupby('match_id').size().reset_index(name='ohl_goals_conceded')

# Combine into one match-level dataframe
match_df = (
    match_meta
    .merge(final_scores, on='match_id', how='left')
    .merge(ohl_goals,    on='match_id', how='left')
    .merge(opp_goals,    on='match_id', how='left')
)
match_df[['ohl_goals_scored', 'ohl_goals_conceded']] = (
    match_df[['ohl_goals_scored', 'ohl_goals_conceded']].fillna(0).astype(int)
)

# Derive result: W / D / L from OHL's perspective
def get_result(row):
    if row['goals_home_ft'] > row['goals_away_ft']:
        return 'W'
    elif row['goals_home_ft'] == row['goals_away_ft']:
        return 'D'
    else:
        return 'L'

match_df['result'] = match_df.apply(get_result, axis=1)

print(match_df.shape)
match_df.head()

(129, 11)


,match_id,match_date,kickoff_time_local,season,stage,opponent,goals_home_ft,goals_away_ft,ohl_goals_scored,ohl_goals_conceded,result
0,3hphnm42ozcm1yye5nocz3ytw,2023-07-29,18:15:00,2023/2024,Regular Season,Sporting Charleroi,1,1,2,0,D
1,3kewmr690bhoid47mku9wfgno,2023-08-05,20:45:00,2023/2024,Regular Season,RWDM,1,2,3,0,L
2,3m8rcryxrola5kngcoiekflec,2023-08-12,18:15:00,2023/2024,Regular Season,Union Saint-Gilloise,5,1,6,0,W
3,3nrkjdhpc5zse1sdzt1onor2s,2023-08-18,20:45:00,2023/2024,Regular Season,Antwerp,1,1,2,0,D
4,3qd97tj5mca89c1m61vh5j190,2023-08-26,18:15:00,2023/2024,Regular Season,Eupen,3,1,4,0,W


In [10]:
# match_df.to_csv('../../data/cleaned/gold_match_goals_cleaned.csv', index=False)
# print('Saved.')